In [ ]:
!pip install "chronos-forecasting @ git+https://github.com/amazon-science/chronos-forecasting.git" --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon
import time
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# ДИПЛОМНАЯ РАБОТА: Прогнозирование дебиторской задолженности
# ============================================================

!pip install neuralforecast datasetsforecast --quiet
!pip install statsforecast tslearn scikit-posthocs statsmodels --quiet



from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA, AutoETS, AutoCES, AutoRegressive, MSTL,
    HoltWinters, AutoTheta, ADIDA, CrostonClassic,
    CrostonOptimized, CrostonSBA, IMAPA,
    SimpleExponentialSmoothing, SimpleExponentialSmoothingOptimized,
    Naive, SeasonalNaive
)

from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoNHITS, AutoPatchTST, AutoTimesNet
from neuralforecast.models import NHITS, NBEATS, TFT, TiDE, DeepAR

from utilsforecast.losses import smape, mase, rmsse
from utilsforecast.evaluation import evaluate
from functools import partial



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 73.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.
     ━━━━━

Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x78e465f5cb80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: /usr/local/lib/python3.12/dist-packages/scipy.libs/libscipy_openblas-b75cc656.

In [ ]:
# Константы
HORIZON = 4
split_date = pd.to_datetime("2023-09-01")

In [ ]:
# ------------------------------------------------------------------
# 1. ЗАГРУЗКА И ПРЕДОБРАБОТКА ДАННЫХ
# ------------------------------------------------------------------
def load_and_preprocess(file_path):
    """Загрузка и предобработка данных"""
    df = pd.read_excel(file_path)
    df = df.drop_duplicates()

    # Создание unique_id
    df["unique_id"] = (
        df["id"].astype(str) + "_" +
        df["Код филиала"].astype(str) + "_" +
        df["Код ЦФО"].astype(str)
    )

    # Берём оба показателя (ДЗ н.п. и ДЗ к.п.)
    df_both = df[df["Показатель"].isin(["ДЗ н.п.", "ДЗ к.п."])]

    # Преобразование в длинный формат
    id_vars = ["id", "Код филиала", "Код ЦФО", "Показатель", "number", "unique_id"]
    df_long = pd.melt(df_both, id_vars=id_vars, var_name="month", value_name="y")

    # Функция для корректного создания дат
    def get_date(row):
        month_num = int(row['month'])
        if month_num <= 12:
            year, month = 2022, month_num
        else:
            year, month = 2023, month_num - 12

        if row['Показатель'] == 'ДЗ н.п.':
            return pd.Timestamp(f"{year}-{month:02d}-01")
        else:
            if month == 12:
                next_year, next_month = year + 1, 1
            else:
                next_year, next_month = year, month + 1
            next_month_start = pd.Timestamp(f"{next_year}-{next_month:02d}-01")
            return next_month_start - pd.Timedelta(days=1)

    df_long['ds'] = df_long.apply(get_date, axis=1)

    # Фильтрация нулевых рядов
    mask = df_long.groupby("unique_id")["y"].transform("sum") != 0
    df_long = df_long[mask].sort_values(["unique_id", "ds"])

    # Создание объединенного ряда (ДЗ н.п. + ДЗ к.п. со сдвигом)
    df_start = df_long[df_long['Показатель'] == 'ДЗ н.п.'].copy()
    df_end = df_long[df_long['Показатель'] == 'ДЗ к.п.'].copy()
    df_end['ds'] = df_end['ds'] + pd.offsets.MonthBegin(1)
    df_combined = pd.concat([df_start, df_end], ignore_index=True)
    df_combined = df_combined.sort_values(['unique_id', 'ds'])
    df_combined = df_combined.drop_duplicates(subset=['unique_id', 'ds'], keep='first')


    return df_combined[['unique_id', 'y', 'ds']]

In [ ]:
# ------------------------------------------------------------------
# 2. КЛАССИФИКАЦИЯ РЯДОВ ПО МЕТОДИКЕ SYNTETOS/BOYLAN (ADI/CV²)
# ------------------------------------------------------------------

def classify_time_series(df_long):
    """Классификация временных рядов по ADI и CV²"""
    results = []

    for uid, group in df_long.groupby('unique_id'):
        T = len(group)
        non_zero_values = group[group['y'] > 0]['y']
        N = len(non_zero_values)

        # ADI (Average Demand Interval)
        if N == 0:
            adi = np.inf
        else:
            adi = T / N

        # CV² (Coefficient of Variation squared)
        if N == 0:
            cv2 = np.nan
        elif N == 1:
            cv2 = 0.0
        else:
            mean = non_zero_values.mean()
            std = non_zero_values.std(ddof=1)
            cv = std / mean if mean != 0 else np.inf
            cv2 = cv ** 2

        # Классификация
        adi_threshold, cv2_threshold = 1.32, 0.49

        if adi <= adi_threshold and cv2 <= cv2_threshold:
            demand_class = "Smooth"
        elif adi <= adi_threshold and cv2 > cv2_threshold:
            demand_class = "Erratic"
        elif adi > adi_threshold and cv2 <= cv2_threshold:
            demand_class = "Intermittent"
        elif adi > adi_threshold and cv2 > cv2_threshold:
            demand_class = "Lumpy"
        else:
            demand_class = "Undefined"

        results.append({'unique_id': uid, 'adi': adi, 'cv2': cv2, 'class': demand_class})

    return pd.DataFrame(results)



In [ ]:
# ------------------------------------------------------------------
# 3. РАЗДЕЛЕНИЕ НА ТРЕНИРОВОЧНУЮ И ТЕСТОВУЮ ВЫБОРКИ
# ------------------------------------------------------------------
def split_train_test(df_long, split_date):
    train = df_long[df_long['ds'] <= split_date].copy()
    test = df_long[df_long['ds'] > split_date].copy()
    return train, test

In [ ]:
# ------------------------------------------------------------------
# Фильтрация датасета по классам
# ------------------------------------------------------------------
def filter_by_class(df, classification_df, class_name):
    """Фильтрует DataFrame (train или test) по указанному классу."""
    ids = classification_df[classification_df['class'] == class_name]['unique_id'].values
    return df[df['unique_id'].isin(ids)].copy()

In [ ]:
# ------------------------------------------------------------------
# 4. Определение списков моделей (статистические, нейросетевые ручные, нейросетевые авто)
# ------------------------------------------------------------------

STAT_MODELS = [
    AutoARIMA(), AutoETS(), AutoCES(),
    AutoRegressive(lags=[1]),
    MSTL(season_length=[3,12], trend_forecaster=AutoARIMA()),
    HoltWinters(season_length=12),
    AutoTheta(season_length=12),
    ADIDA(), CrostonClassic(), CrostonOptimized(), CrostonSBA(), IMAPA(),
    SimpleExponentialSmoothing(alpha=0.5),
    SimpleExponentialSmoothingOptimized()
]

MANUAL_NN_MODELS = [
    NHITS(h=HORIZON, input_size=16, scaler_type='robust', max_steps=500,
          early_stop_patience_steps=-1, accelerator='auto', random_seed=42),
    NBEATS(h=HORIZON, input_size=16, scaler_type='robust', max_steps=500,
           early_stop_patience_steps=-1, accelerator='auto', random_seed=42),
    TFT(h=HORIZON, input_size=16, scaler_type='robust', max_steps=500,
        early_stop_patience_steps=-1, accelerator='auto', random_seed=42),
    TiDE(h=HORIZON, input_size=16, scaler_type='robust', max_steps=500,
         early_stop_patience_steps=-1, accelerator='auto', random_seed=42),
    DeepAR(h=HORIZON, input_size=16, scaler_type='robust', max_steps=500,
           early_stop_patience_steps=-1, accelerator='auto', random_seed=42)
]

AUTO_NN_MODELS = [
    AutoNHITS(h=HORIZON),
    AutoPatchTST(h=HORIZON),
    AutoTimesNet(h=HORIZON)
]

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:lightning_fabric.utilities.seed:Seed set to 42


In [ ]:
import torch
from chronos import ChronosPipeline

# Словарь с тремя версиями Chronos
chronos_models = {
    "Chronos_tiny": ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-tiny", # 8M параметров, летает даже на CPU
        device_map="cuda" if torch.cuda.is_available() else "cpu",
        torch_dtype=torch.bfloat16,
    ),
    "Chronos_mini": ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-mini", # 20M параметров, CPU / GPU
        device_map="cuda" if torch.cuda.is_available() else "cpu",
        torch_dtype=torch.bfloat16,
    ),
    "Chronos_small": ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-small", # 46M параметров, рекомендуется GPU
        device_map="cuda" if torch.cuda.is_available() else "cpu",
        torch_dtype=torch.bfloat16,
    ),
}

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/81.8M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/185M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

In [ ]:
def forecast_chronos_multi(train_df, test_df, chronos_dict, horizon=4):
    """
    Прогноз для нескольких версий Chronos.

    Параметры:
    ----------
    train_df, test_df : DataFrame
        Тренировочные и тестовые данные.
    chronos_dict : dict
        Словарь вида {"имя_модели": pipeline}.
    horizon : int
        Горизонт прогноза.

    Возвращает:
    ----------
    DataFrame с колонками ['unique_id','ds'] + имена моделей из chronos_dict.
    """
    future_dates = test_df[['unique_id','ds']].drop_duplicates().copy()
    # Сразу готовим колонки для всех моделей
    for name in chronos_dict.keys():
        future_dates[name] = np.nan

    for uid in train_df['unique_id'].unique():
        hist = train_df[train_df['unique_id']==uid]['y'].values
        context = torch.tensor(hist, dtype=torch.float32).unsqueeze(0)

        # Маска для строк этого ряда
        mask = future_dates['unique_id'] == uid

        for name, pipeline in chronos_dict.items():
            forecast = pipeline.predict(context, prediction_length=horizon)
            if isinstance(forecast, torch.Tensor):
                values = forecast.detach().cpu().numpy().flatten().tolist()
            else:
                values = np.array(forecast).flatten().tolist()
            values = values[:horizon]

            # Присваиваем значения только для строк этого ряда
            future_dates.loc[mask, name] = values

    return future_dates.reset_index(drop=True)

In [ ]:
# ------------------------------------------------------------------
# 5. Функция обучения и оценки для одного набора данных
# ------------------------------------------------------------------

from statsforecast import StatsForecast
from neuralforecast import NeuralForecast
import time
import pandas as pd

def train_and_evaluate(train_df, test_df, dataset_name,
                       do_stat=True, do_manual_nn=True, do_auto_nn=True, do_FM=True):
    """
    Обучает модели на train_df, делает прогноз на test_df,
    возвращает словарь с forecast DataFrame и метриками.
    """
    results = {}

    # 1. Статистические модели
    if do_stat:
        print(f"[{dataset_name}] Обучение статистических моделей...")
        start = time.time()
        sf = StatsForecast(models=STAT_MODELS, freq='MS', n_jobs=-1)
        forecast_stat = sf.forecast(df=train_df, h=HORIZON)
        forecast_stat = test_df.merge(forecast_stat, on=['unique_id','ds'], how='left')
        results['stat_forecast'] = forecast_stat
        # метрики считаем позже единообразно
        print(f"  готово за {time.time()-start:.1f} сек")

    # 2. Ручные нейросети
    if do_manual_nn:
        print(f"[{dataset_name}] Обучение ручных нейросетей...")
        start = time.time()
        nf = NeuralForecast(models=MANUAL_NN_MODELS, freq='MS')
        nf.fit(df=train_df)
        forecast_manual = nf.predict()
        forecast_manual = test_df.merge(forecast_manual, on=['unique_id','ds'], how='left')
        results['manual_forecast'] = forecast_manual
        print(f"  готово за {time.time()-start:.1f} сек")

    # 3. Автоматические нейросети
    if do_auto_nn:
        print(f"[{dataset_name}] Обучение автоматических нейросетей...")
        start = time.time()
        nf_auto = NeuralForecast(models=AUTO_NN_MODELS, freq='MS')
        nf_auto.fit(df=train_df)
        forecast_auto = nf_auto.predict()
        forecast_auto = test_df.merge(forecast_auto, on=['unique_id','ds'], how='left')
        results['auto_forecast'] = forecast_auto
        print(f"  готово за {time.time()-start:.1f} сек")


    # 4. Автоматические нейросети
    if do_FM:
       print(f"[{dataset_name}] Обучение foundation-моделей Chronos (tiny, mini, small)")
       start = time.time()
       chronos_forecast = forecast_chronos_multi(
           train_df, test_df, chronos_models, HORIZON
       )
       results['chronos_forecast'] = chronos_forecast
       print(f"  готово за {time.time()-start:.1f} сек")

    return results

In [ ]:
# ------------------------------------------------------------------
# 6. Функция оценки метрик и объединения
# ------------------------------------------------------------------

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import smape, mase, rmsse, rmse, cfe
from functools import partial

METRICS = [
    smape,
    partial(mase, seasonality=12),
    partial(rmsse, seasonality=12)
]

def compute_metrics(forecast_df, train_df, model_names):
    """Возвращает eval_df (длинный), median_df, mean_df."""
    eval_df = evaluate(forecast_df, metrics=METRICS, train_df=train_df, models=model_names)
    median_df = eval_df.drop(columns='unique_id').groupby('metric').median().reset_index()
    mean_df = eval_df.drop(columns='unique_id').groupby('metric').mean().reset_index()
    return eval_df, median_df, mean_df

In [ ]:
# ------------------------------------------------------------------
# 7. Основной цикл по классам
# ------------------------------------------------------------------

# Загружаем данные один раз
df_long = load_and_preprocess("data.xlsx")
classification = classify_time_series(df_long)
train_all, test_all = split_train_test(df_long, pd.to_datetime("2023-09-01"))

# Словарь для хранения всех результатов
all_results = {}

# Наборы данных для анализа
datasets = {
    "all": (train_all, test_all),
    "Smooth": (filter_by_class(train_all, classification, "Smooth"),
               filter_by_class(test_all, classification, "Smooth")),
    "Intermittent": (filter_by_class(train_all, classification, "Intermittent"),
                     filter_by_class(test_all, classification, "Intermittent")),
    "Erratic": (filter_by_class(train_all, classification, "Erratic"),
                filter_by_class(test_all, classification, "Erratic")),
    "Lumpy": (filter_by_class(train_all, classification, "Lumpy"),
              filter_by_class(test_all, classification, "Lumpy"))
}



In [ ]:
for name, (train_df, test_df) in datasets.items():
    print(f"\n===== Обработка набора: {name} ({len(train_df['unique_id'].unique())} рядов) =====")
    # Обучаем модели (можно включать/выключать по необходимости)
    res = train_and_evaluate(train_df, test_df, name,
                             do_stat=True, do_manual_nn=True, do_auto_nn=True, do_FM=True)

    # Объединяем прогнозы в один DataFrame
    forecast_full = test_df[['unique_id','ds','y']].copy()
    if 'stat_forecast' in res:
        forecast_full = forecast_full.merge(res['stat_forecast'], on=['unique_id','ds','y'], how='left')
    if 'manual_forecast' in res:
        forecast_full = forecast_full.merge(res['manual_forecast'], on=['unique_id','ds','y'], how='left')
    if 'auto_forecast' in res:
        forecast_full = forecast_full.merge(res['auto_forecast'], on=['unique_id','ds','y'], how='left')
    if 'chronos_forecast' in res:
        forecast_full = forecast_full.merge(res['chronos_forecast'], on=['unique_id','ds'], how='left')

    # Список всех моделей (колонки, кроме unique_id, ds, y)
    model_cols = [c for c in forecast_full.columns if c not in ['unique_id','ds','y']]

    # Оценка
    eval_df, median_df, mean_df = compute_metrics(forecast_full[['unique_id','ds','y']+model_cols],
                                                  train_df, model_cols)   # train_df для нормализации MASE/RMSSE

    # Сохраняем
    all_results[name] = {
        'forecast': forecast_full,
        'eval': eval_df,
        'median': median_df,
        'mean': mean_df
    }

    # Сохраняем в CSV
    forecast_full.to_csv(f'forecasts_{name}.csv', index=False)
    eval_df.to_csv(f'evaluation_{name}.csv', index=False)


===== Обработка набора: all (1431 рядов) =====
[all] Обучение статистических моделей...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


  готово за 771.7 сек
[all] Обучение ручных нейросетей...


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.4 M  | train
-------------------------------------------------------
2.4 M     Trainable params
0         Non-trainable params
2.4 M     Total params
9.640     Total estimated model params size (MB)
34        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

  готово за 3488.7 сек
[Smooth] Обучение foundation-моделей Chronos (tiny, mini, small)
  готово за 99.9 сек

===== Обработка набора: Intermittent (513 рядов) =====
[Intermittent] Обучение статистических моделей...
